# Generating an audit report — SUPPORT2 ICU mortality

This is the example script the *isitfair* manuscript refers to. It runs a
complete subgroup deployment-readiness audit of an in-hospital mortality model
and writes a self-contained TRIPOD+AI HTML report.

**If you want to audit your own model, start here.** Replace section 2 with
your own outcome, predictions and subgroup labels; the rest runs unchanged.

## The cohort

SUPPORT2 (Study to Understand Prognoses and Preferences for Outcomes and Risks
of Treatment) is 9,105 critically ill adults from five US medical centres,
distributed by the UCI Machine Learning Repository under CC BY 4.0 with no
credentialing. The outcome is in-hospital mortality (`hospdead`).

The audit runs on a held-out **test partition of n = 1,821**. A separate
calibration partition is held back so recalibration is fitted on data the
audit never sees — the leak-free arrangement any mitigation comparison needs.

The three audit axes are **sex**, **age group** and **race/ethnicity**.
SUPPORT2 records all three, so a single audit exercises a sociodemographic
axis, a clinical one, and the intersection of two of them.

Every random operation is seeded, so re-running this reproduces the shipped
report exactly.

## Running it

```bash
pip install -e ".[examples]"
python examples/support2_mortality_audit.py          # ~12 min at B = 2000
ISITFAIR_FAST=1 python examples/support2_mortality_audit.py   # ~1 min at B = 200
```

## 1. Setup

In [1]:
import importlib.util
import sys
from pathlib import Path

# Make examples/ importable whether this runs as a script or as the paired notebook.
_here = Path.cwd()
for _candidate in (_here, *_here.parents):
    if (_candidate / "examples" / "support2_data.py").exists():
        sys.path.insert(0, str(_candidate / "examples"))
        break

import isitfair
from isitfair import FairnessAudit, GroupRecalibration
from support2_data import (
    SEED,
    THRESHOLD,
    build_model_matrix,
    fit_model,
    load_support2,
    make_sensitive_attributes,
    n_bootstrap,
    repo_root,
    results_dir,
    split_cohort,
)

B = n_bootstrap()
RESULTS = results_dir()

print(f"isitfair {isitfair.__version__}")
print(f"seed {SEED} · threshold {THRESHOLD} · bootstrap replicates {B}")
print(f"results -> {RESULTS.relative_to(repo_root())}/")

isitfair 0.4.0
seed 99 · threshold 0.3 · bootstrap replicates 2000
results -> results/


## 2. The four things you must bring

An audit needs four inputs. The package works from predictions, so your
feature matrix stays with you.

| Input | Here |
|---|---|
| `y_true` — binary outcome | in-hospital mortality |
| `y_proba` — predicted risk, **out of fold** | logistic regression fitted on a disjoint training partition |
| `sensitive_features` — subgroup labels | sex, age group, race/ethnicity |
| `threshold` — a prespecified operating point | 0.30 |

The predictions must be out of fold. In-sample predictions are optimistic,
and *differentially* so: the optimism is largest in the sparsest subgroups,
which is exactly where an audit is trying to look.

Below, each stage of the pipeline is one call. The implementations live in
`examples/support2_data.py` so that this script, the paper-analysis notebook
and the tutorial notebook all audit an identical test partition.

In [2]:
df = load_support2()
print(f"N = {len(df)}   in-hospital mortality = {df['hospdead'].mean():.3f}")

N = 9105   in-hospital mortality = 0.259


In [3]:
# Sex and race are audit axes; the design matrix is built without them.
attrs = make_sensitive_attributes(df)
for name, values in attrs.items():
    import numpy as np

    labels, counts = np.unique(values, return_counts=True)
    print(f"{name:>15}: " + ", ".join(f"{lab} n={cnt}" for lab, cnt in zip(labels, counts)))

            sex: female n=3980, male n=5125
      age_group: 18-44 n=1357, 45-64 n=3235, 65-79 n=3467, 80+ n=1046
 race_ethnicity: asian n=79, black n=1391, hispanic n=290, other n=154, white n=7191


In [4]:
X = build_model_matrix(df)
y = df["hospdead"].to_numpy().astype(float)
print(f"design matrix: {X.shape[1]} columns, {int(X.isnull().sum().sum())} missing")
print(f"sex in features?  {'sex' in X.columns}")
print(f"race in features? {'race' in X.columns}")

design matrix: 18 columns, 0 missing
sex in features?  False
race in features? False


In [5]:
idx_train, idx_calib, idx_test = split_cohort(X, y, seed=SEED)
model, scaler = fit_model(X, y, idx_train, seed=SEED)


def predict(idx):
    return model.predict_proba(scaler.transform(X.iloc[idx]))[:, 1]


y_test, p_test = y[idx_test], predict(idx_test)
y_calib, p_calib = y[idx_calib], predict(idx_calib)
attrs_test = {k: v[idx_test] for k, v in attrs.items()}
attrs_calib = {k: v[idx_calib] for k, v in attrs.items()}

for label, part in (("train", idx_train), ("calib", idx_calib), ("test", idx_test)):
    print(f"{label:>5}: n={len(part):>5}  events={int(y[part].sum()):>4}  prev={y[part].mean():.3f}")

train: n= 5463  events=1416  prev=0.259
calib: n= 1821  events= 472  prev=0.259
 test: n= 1821  events= 472  prev=0.259


## 3. Constructing the audit

`FairnessAudit` is lazy: the bootstrap runs the first time you ask for an
axis, and every axis afterwards reuses the same replicate index array. That is what makes a gap statistic a *reduction over*
the per-subgroup distributions rather than an independent resample of them.

In [6]:
audit = FairnessAudit(
    y_true=y_test,
    y_proba=p_test,
    sensitive_features=attrs_test,
    threshold=THRESHOLD,
    n_bootstrap=B,
    random_state=SEED,
)
print(audit)

## 4. Discrimination — can the model tell them apart?

AUROC and AUPRC are threshold-free; the rates below them are properties of
the decision rule at t = 0.30. Note that AUPRC's no-skill baseline is each
subgroup's own prevalence, not 0.5, so ranking subgroups by AUPRC largely
ranks their prevalences.

In [7]:
disc = audit.discrimination()
cols = ["attribute", "subgroup", "n", "n_events", "prevalence", "auroc", "auroc_ci_low", "auroc_ci_high"]
print(disc[disc.subgroup != "GAP"][cols].to_string(index=False))

     attribute subgroup      n  n_events  prevalence    auroc  auroc_ci_low  auroc_ci_high
           sex  Overall 1821.0     472.0    0.259198 0.846028      0.825305       0.865933
           sex   female  787.0     199.0    0.252859 0.833248      0.800488       0.863965
           sex     male 1034.0     273.0    0.264023 0.856142      0.828934       0.880544
     age_group  Overall 1821.0     472.0    0.259198 0.846028      0.825305       0.865933
     age_group    18-44  257.0      65.0    0.252918 0.878686      0.828753       0.921541
     age_group    45-64  684.0     171.0    0.250000 0.860196      0.826049       0.892116
     age_group    65-79  677.0     172.0    0.254062 0.829277      0.794705       0.860208
     age_group      80+  203.0      64.0    0.315271 0.811488      0.741168       0.872815
race_ethnicity  Overall 1821.0     472.0    0.259198 0.846028      0.825305       0.865933
race_ethnicity    asian   13.0       5.0    0.384615 0.800000           NaN            NaN

### The four parity gaps

Equal opportunity (sensitivity), predictive equality (FPR), statistical parity
(flag rate) and equalized odds (the max of the first two). All four are
properties of the decision rule at one threshold — move the threshold and they
all move.

Read each gap next to the within-group numbers above it: parity is equally
consistent with both groups being served well and with both being served
badly, and only the within-group rows tell you which. Every frame carries
both.

In [8]:
gaps = disc[disc.subgroup == "GAP"][["attribute", "gap_metric", "gap_value", "gap_ci_low", "gap_ci_high"]]
print(gaps.to_string(index=False))

     attribute                   gap_metric  gap_value  gap_ci_low  gap_ci_high
           sex equal_opportunity_difference   0.042299    0.002247     0.126338
           sex          predictive_equality   0.031088    0.002134     0.072754
           sex           equalized_odds_gap   0.042299    0.013125     0.126338
           sex           statistical_parity   0.028077    0.001696     0.071432
     age_group equal_opportunity_difference   0.077485    0.039733     0.215690
     age_group          predictive_equality   0.130941    0.072448     0.210696
     age_group           equalized_odds_gap   0.130941    0.091557     0.227222
     age_group           statistical_parity   0.150095    0.078200     0.225624
race_ethnicity equal_opportunity_difference   0.104918         NaN          NaN
race_ethnicity          predictive_equality   0.163043         NaN          NaN
race_ethnicity           equalized_odds_gap   0.163043         NaN          NaN
race_ethnicity           statistical_par

### ROC curves

Per-subgroup ROC on a fixed false-positive-rate grid with bootstrap bands.
Where curves cross, the ranking between subgroups changes with the operating
point — detail a single summary AUROC cannot show.

In [9]:
roc = audit.roc_curves()
print(f"{roc.shape[0]} rows across {roc['attribute'].nunique()} axes")

1400 rows across 3 axes


## 5. Calibration — is the number right?

Calibration is where subgroup disparities most often hide in clinical models.
A model that never saw race as a feature can still be miscalibrated across
racial groups, because the features it *did* see are distributed differently
(Obermeyer et al., *Science* 2019).

Read the intercept and slope together. The intercept α is
calibration-in-the-large with the slope pinned at 1; it is prevalence
dependent, so it is only interpretable against a stated population. The slope
β is fitted with both terms free: β < 1 means the predictions are spread too
wide, β > 1 that they are too compressed.

In [10]:
cal = audit.calibration()
print(
    cal[["attribute", "subgroup", "n", "calibration_intercept", "calibration_slope", "brier", "ece", "ici"]]
    .to_string(index=False)
)

     attribute subgroup      n  calibration_intercept  calibration_slope    brier      ece      ici
           sex  Overall 1821.0               0.082409           1.033646 0.129298 0.019082 0.013658
           sex   female  787.0              -0.029882           1.007830 0.130851 0.022756 0.009457
           sex     male 1034.0               0.171066           1.062549 0.128117 0.034439 0.026475
           sex      GAP    NaN                    NaN                NaN      NaN      NaN      NaN
           sex      GAP    NaN                    NaN                NaN      NaN      NaN      NaN
     age_group  Overall 1821.0               0.082409           1.033646 0.129298 0.019082 0.013658
     age_group    18-44  257.0              -0.025151           1.173759 0.112402 0.033332 0.019216
     age_group    45-64  684.0               0.267405           1.113330 0.121203 0.037952 0.031088
     age_group    65-79  677.0              -0.049054           0.962521 0.136564 0.035703 0.025771


In [11]:
curves = audit.calibration_curves()
print(f"LOESS calibration curves: {curves.shape[0]} rows across {curves['attribute'].nunique()} axes")

LOESS calibration curves: 1400 rows across 3 axes


### Hosmer–Lemeshow

Reported with everything Kramer & Zimmerman (2007) require alongside it: the
subgroup's *n*, the full per-decile observed-versus-expected table, and the
aggregate scores above.

Read each statistic beside its own *n*. The value scales with sample size, so
it compares a subgroup against its own expected counts rather than against
another subgroup. And a large p-value means miscalibration was not detected,
which in a small subgroup is the expected outcome either way — the decile
table below is what carries the detail.

`df_mode` defaults to `"validation"`, i.e. df = g rather than g − 2, because
out-of-fold predictions estimate nothing on this data. Using g − 2 here
roughly doubles the false miscalibration rate.

In [12]:
hl = audit.hosmer_lemeshow()
print(
    hl[hl.subgroup != "GAP"][
        ["attribute", "subgroup", "n", "n_events", "hl_statistic", "hl_df",
         "hl_p_value", "n_bins_used", "bins_reduced", "decile_slope", "decile_intercept"]
    ].to_string(index=False)
)

     attribute subgroup      n  n_events  hl_statistic  hl_df  hl_p_value  n_bins_used  bins_reduced  decile_slope  decile_intercept
           sex  Overall 1821.0     472.0      6.984481   10.0    0.726909         10.0         False      1.035193          0.001901
           sex   female  787.0     199.0      4.889447   10.0    0.898440         10.0         False      0.999816         -0.003906
           sex     male 1034.0     273.0     11.609940   10.0    0.312009         10.0         False      1.074030          0.003854
     age_group  Overall 1821.0     472.0      6.984481   10.0    0.726909         10.0         False      1.035193          0.001901
     age_group    18-44  257.0      65.0      3.455898   10.0    0.968575         10.0         False      1.060467         -0.018412
     age_group    45-64  684.0     171.0      9.862405   10.0    0.452647         10.0         False      1.113864          0.007580
     age_group    65-79  677.0     172.0      9.413592   10.0    0.49

In [13]:
hl_deciles = audit.hosmer_lemeshow(detail=True)
print(f"per-decile observed vs expected: {hl_deciles.shape[0]} rows")
print(hl_deciles.head(10).to_string(index=False))

per-decile observed vs expected: 140 rows
attribute subgroup  decile     n  observed   expected  observed_rate  expected_rate  difference
      sex  Overall       1 183.0       4.0   4.540307       0.021858       0.024810   -0.540307
      sex  Overall       2 182.0       6.0   7.731616       0.032967       0.042481   -1.731616
      sex  Overall       3 182.0      11.0  10.941175       0.060440       0.060116    0.058825
      sex  Overall       4 182.0      17.0  15.428930       0.093407       0.084774    1.571070
      sex  Overall       5 182.0      20.0  22.806125       0.109890       0.125308   -2.806125
      sex  Overall       6 182.0      42.0  33.256611       0.230769       0.182729    8.743389
      sex  Overall       7 182.0      50.0  47.405714       0.274725       0.260471    2.594286
      sex  Overall       8 182.0      76.0  68.087159       0.417582       0.374105    7.912841
      sex  Overall       9 182.0      94.0  96.603701       0.516484       0.530790   -2.60370

## 6. Clinical utility — is it worth acting on?

A decision curve asks whether acting on the model beats the two strategies
that need no model at all: treat everyone, or treat no one. Net benefit
subtracts false positives at the exchange rate the threshold implies —
t/(1−t), which at t = 0.30 is about 0.43. That ratio is arithmetic from the
threshold, not an independently elicited preference.

The treat-all reference uses **each subgroup's own prevalence**, so every
subgroup is compared against the no-model strategy as it would actually
perform in that subgroup.

In [14]:
dc = audit.decision_curve(zero_crossing=True)
print(
    dc[dc.subgroup != "GAP"][
        ["attribute", "subgroup", "threshold", "net_benefit", "standardized_net_benefit", "treat_all_nb"]
    ].head(12).to_string(index=False)
)

attribute subgroup  threshold  net_benefit  standardized_net_benefit  treat_all_nb
      sex  Overall       0.02     0.244584                  0.943618      0.244080
      sex  Overall       0.04     0.231031                  0.891331      0.228332
      sex  Overall       0.06     0.219239                  0.845835      0.211913
      sex  Overall       0.08     0.208032                  0.802598      0.194781
      sex  Overall       0.10     0.198182                  0.764595      0.176887
      sex  Overall       0.12     0.191254                  0.737866      0.158180
      sex  Overall       0.14     0.184616                  0.712259      0.138603
      sex  Overall       0.16     0.176015                  0.679076      0.118093
      sex  Overall       0.18     0.162401                  0.626550      0.096583
      sex  Overall       0.20     0.155821                  0.601165      0.073998
      sex  Overall       0.22     0.148974                  0.574750      0.050254
    

## 7. How much of this is noise?

`estimability()` is the single gate, and it answers the question in one
table: which estimates this cohort supports an interval for. A cell is
reportable when it clears **both** rules — the pre-registered event floor for
that metric, and the bootstrap replicate rule (at most 10% of replicates
non-estimable). A gap inherits the verdict of both constituent subgroups, and
suppression withholds the **interval, keeping the point estimate**: the value
is real, it simply has no defensible uncertainty around it.

In [15]:
est = audit.estimability()
unreportable = est[est.not_estimable_final]
print(f"{len(unreportable)} of {len(est)} cell-metrics are not reportable")
print(
    unreportable[["attribute", "subgroup", "metric", "n", "n_events", "event_floor", "below_floor", "proportion_nonestimable"]]
    .to_string(index=False)
)

41 of 178 cell-metrics are not reportable
     attribute subgroup                       metric     n  n_events  event_floor  below_floor  proportion_nonestimable
     age_group    18-44            calibration_slope 257.0      65.0        100.0         True                   0.0000
     age_group      80+            calibration_slope 203.0      64.0        100.0         True                   0.0000
     age_group      GAP                max_slope_gap   NaN       NaN        100.0         True                   0.0000
race_ethnicity      GAP equal_opportunity_difference   NaN       NaN         25.0         True                   0.0000
race_ethnicity      GAP           equalized_odds_gap   NaN       NaN         25.0         True                   0.0000
race_ethnicity      GAP            max_intercept_gap   NaN       NaN         25.0         True                   0.0000
race_ethnicity      GAP          max_net_benefit_gap   NaN       NaN         50.0         True                      Na

## 8. Intersectional cells

Passing a tuple key builds the interaction. Sex × race/ethnicity gives 10
cells across a wide range of sizes, which is exactly the case empirical-Bayes
shrinkage exists for: cells below the event threshold are shrunk toward the
first constituent's marginal rates, only rate metrics are shrunk (AUROC and
AUPRC keep their own values), and shrunken cells carry NaN intervals so the
borrowing is visible.

In [16]:
audit_inter = FairnessAudit(
    y_true=y_test,
    y_proba=p_test,
    sensitive_features={**attrs_test, ("sex", "race_ethnicity"): None},
    threshold=THRESHOLD,
    n_bootstrap=B,
    random_state=SEED,
)
cells = audit_inter.cell_diagnostics()
print(cells.to_string(index=False))
print(f"\n{int(cells['shrunk'].sum())} of {len(cells)} cells were shrunk")

           attribute          subgroup   n  n_events  shrunk  marginal_rate  cell_rate  shrinkage_factor
sex × race_ethnicity    female / asian   2         2    True       0.252859   1.000000          0.988038
sex × race_ethnicity    female / black 127        40    True       0.252859   0.314961          0.565366
sex × race_ethnicity female / hispanic  24         3    True       0.252859   0.125000          0.873150
sex × race_ethnicity    female / other  14         6    True       0.252859   0.428571          0.921875
sex × race_ethnicity    female / white 620       148   False       0.252859   0.238710          0.210392
sex × race_ethnicity      male / asian  11         3    True       0.264023   0.272727          0.937571
sex × race_ethnicity      male / black 139        42    True       0.264023   0.302158          0.543064
sex × race_ethnicity   male / hispanic  33         7    True       0.264023   0.212121          0.833502
sex × race_ethnicity      male / other  18         3   

## 9. Mitigation

Group-specific isotonic recalibration, fitted on the held-back calibration
partition. Isotonic is a monotone step function, so it preserves rank within
a subgroup — AUROC is untouched by construction — and only moves the numbers.

The check that matters is **levelling down** (Pfohl et al. 2021): confirm the
disadvantaged group actually improved, rather than the well-calibrated groups
converging downward to meet it.

In [17]:
audit_pre = FairnessAudit(
    y_true=y_test,
    y_proba=p_test,
    sensitive_features=attrs_test,
    threshold=THRESHOLD,
    n_bootstrap=B,
    random_state=SEED,
)
mitigated = audit_pre.mitigate(
    method=GroupRecalibration(method="isotonic"),
    calibration_data=(y_calib, p_calib, attrs_calib),
)

keep = ["subgroup", "calibration_intercept", "calibration_slope", "brier", "ece"]
before = audit_pre.calibration()
after = mitigated["race_ethnicity"].calibration()
print("BEFORE (race_ethnicity):")
print(before[before.attribute == "race_ethnicity"][keep].to_string(index=False))
print("\nAFTER group-specific isotonic recalibration:")
print(after[after.attribute == "race_ethnicity"][keep].to_string(index=False))

BEFORE (race_ethnicity):
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall               0.082409           1.033646 0.129298 0.019082
   asian               0.674644           0.948987 0.200621 0.348770
   black               0.336408           0.870793 0.157254 0.055203
hispanic              -0.492658           0.947605 0.106005 0.098122
   other               0.431006           0.941341 0.125675 0.123582
   white               0.038975           1.077807 0.124536 0.023804
     GAP                    NaN                NaN      NaN      NaN
     GAP                    NaN                NaN      NaN      NaN

AFTER group-specific isotonic recalibration:
subgroup  calibration_intercept  calibration_slope    brier      ece
 Overall               0.057901           0.628589 0.131329 0.017830
   asian               0.998499           0.647445 0.235401 0.383316
   black               0.246544           0.443329 0.158801 0.062739
hispanic              -0.979672 

## 10. The report

One self-contained HTML file, no JavaScript, with 300-DPI PNGs written to a
sibling `_figures/` directory. It carries all three axes, the gap statistics,
the per-decile H–L tables, the intersectional cell diagnostics, the
before/after mitigation section and a TRIPOD+AI appendix.

`pdf=True` also typesets it as a PDF beside the HTML. That needs the `pdf`
extra (`pip install -e ".[pdf]"`, which installs WeasyPrint), so the cell
asks for the PDF only when WeasyPrint is importable.

In [18]:
write_pdf = importlib.util.find_spec("weasyprint") is not None
report_path = audit.report(
    out=RESULTS / "support2_audit_report.html",
    title="ICU mortality (SUPPORT2) — subgroup deployment-readiness audit",
    metadata={
        "Model": "Logistic regression (13 clinical features + disease class, cancer status)",
        "Cohort": f"SUPPORT2, UCI id 880 (N=9,105); audited on held-out test partition n={len(y_test)}",
        "Outcome": f"In-hospital mortality ({y_test.mean():.1%} in the test partition)",
        "Threshold": str(THRESHOLD),
        "Predictions": "Out of fold — fitted on a disjoint 60% training partition",
        "Uncertainty": f"Patient-level bootstrap, B={B}, random_state={SEED}",
        "isitfair version": isitfair.__version__,
    },
    mitigation_audits=mitigated,
    pdf=write_pdf,
)
print(f"report written to {report_path.relative_to(repo_root())}")
if write_pdf:
    print(f"PDF written to {report_path.with_suffix('.pdf').relative_to(repo_root())}")
else:
    print("PDF skipped: install the pdf extra to write one")

report written to results/support2_audit_report.html
PDF written to results/support2_audit_report.pdf


## 11. Adapting this to your own model

1. Replace section 2 with your data. You need `y_true`, out-of-fold
   `y_proba`, and a dict of subgroup label arrays.
2. Set `threshold` to your clinical operating point, and prespecify it —
   every rate metric and the whole utility axis move with it.
3. Set `event_floors` for your setting if the defaults (AUROC 25,
   calibration intercept 25, slope 100, net benefit 50) do not fit.
4. Run sections 3–10 unchanged.

The next two notebooks go further: `02_paper_analysis_support2.ipynb`
works through every analysis the package supports on this cohort, and
`03_tutorial_support2.ipynb` is the tutorial deck made runnable.

### References

- Knaus WA, et al. The SUPPORT prognostic model. *Ann Intern Med* 1995;122:191–203.
- Obermeyer Z, et al. Dissecting racial bias in an algorithm used to manage
  the health of populations. *Science* 2019;366:447–53.
- Kramer AA, Zimmerman JE. Assessing the calibration of mortality benchmarks
  in critical care. *Crit Care Med* 2007;35:2052–6.
- Pfohl SR, et al. An empirical characterization of fair machine learning for
  clinical risk prediction. *J Biomed Inform* 2021;113:103621.